In [2]:
import pandas as pd
import numpy as np

In [3]:
raw_data = pd.read_csv("industrial-machine-condition-classification/student_dataset.csv")
test = pd.read_csv("industrial-machine-condition-classification/test.csv")

raw_data.head()

,unit_id,unit_type,ambient_temp,process_temp,coolant_flow,shaft_speed,drive_torque,power_load,component_wear,thermal_stress,mechanical_load,vibration_index,condition
0,UNT-184174,M,21.5,56.4,17.85,1454.0,58.2,9.099,30.0,29.31,57.19,1.768,0
1,UNT-236234,H,29.9,65.0,30.88,1720.0,54.6,10.101,13.0,23.93,36.94,1.081,0
2,UNT-247460,M,26.4,47.8,19.75,1552.0,38.4,6.328,89.0,13.52,38.22,1.031,0
3,UNT-176316,M,28.8,60.9,19.10,1513.0,49.1,7.768,59.0,24.62,46.17,1.011,0
4,UNT-105110,S,29.4,64.0,11.77,1536.0,52.6,8.405,36.0,27.83,94.72,1.851,1


In [4]:
X = raw_data.drop(columns=["unit_id", "condition"])
y = raw_data["condition"]

cat_cols = ["unit_type"]
num_cols = X.select_dtypes(include="number").columns.to_list()

print(cat_cols)
print(num_cols)


['unit_type']
['ambient_temp', 'process_temp', 'coolant_flow', 'shaft_speed', 'drive_torque', 'power_load', 'component_wear', 'thermal_stress', 'mechanical_load', 'vibration_index']


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

xgb_num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

xgb_cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

xgb_preprocessor = ColumnTransformer([
    ("num", xgb_num_pipeline, num_cols),
    ("cat", xgb_cat_pipeline, cat_cols)
])

In [6]:
from sklearn.model_selection import StratifiedKFold

stk = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [7]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    random_state=42,
    eval_metric="mlogloss"
)

xgb_pipeline = Pipeline([
    ("preprocessor", xgb_preprocessor),
    ("model", xgb)
])



In [8]:
from sklearn.model_selection import cross_validate


xgb_cv_results = cross_validate(
    xgb_pipeline,
    X,
    y,
    cv=stk,
    scoring="accuracy",
    return_train_score=True
)

print("Train Accuracy:", xgb_cv_results["train_score"])
print("Validation Accuracy:", xgb_cv_results["test_score"])

print("Mean Train Accuracy:", xgb_cv_results["train_score"].mean())
print("Mean Validation Accuracy:", xgb_cv_results["test_score"].mean())
print("Validation Std:", xgb_cv_results["test_score"].std())

Train Accuracy: [0.981875   0.98179688 0.98132812 0.980625   0.9821875 ]
Validation Accuracy: [0.885625  0.8784375 0.8871875 0.8775    0.8784375]
Mean Train Accuracy: 0.9815625000000001
Mean Validation Accuracy: 0.8814375
Validation Std: 0.004101257429130749


In [9]:
X_temp = X.copy()

In [10]:
X_temp.head()

,unit_type,ambient_temp,process_temp,coolant_flow,shaft_speed,drive_torque,power_load,component_wear,thermal_stress,mechanical_load,vibration_index
0,M,21.5,56.4,17.85,1454.0,58.2,9.099,30.0,29.31,57.19,1.768
1,H,29.9,65.0,30.88,1720.0,54.6,10.101,13.0,23.93,36.94,1.081
2,M,26.4,47.8,19.75,1552.0,38.4,6.328,89.0,13.52,38.22,1.031
3,M,28.8,60.9,19.10,1513.0,49.1,7.768,59.0,24.62,46.17,1.011
4,S,29.4,64.0,11.77,1536.0,52.6,8.405,36.0,27.83,94.72,1.851


In [11]:
X_temp["power_ratio"] = (
    X_temp["power_load"] /
    (X_temp["shaft_speed"] * X_temp["drive_torque"])
)

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from xgboost import XGBClassifier


def evaluate_features(X_data, y, skf):
    
    # Sütunları otomatik belirle
    cat_cols = X_data.select_dtypes(
        exclude="number"
    ).columns.to_list()

    num_cols = X_data.select_dtypes(
        include="number"
    ).columns.to_list()

    # Numeric preprocessing
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    # Categorical preprocessing
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Preprocessor
    preprocessor = ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ])

    # Model
    model = XGBClassifier(
        random_state=42,
        eval_metric="mlogloss"
    )

    # Full pipeline
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Cross-validation
    results = cross_validate(
        pipeline,
        X_data,
        y,
        cv=skf,
        scoring="accuracy",
        return_train_score=True
    )

    train_mean = results["train_score"].mean()
    val_mean = results["test_score"].mean()
    val_std = results["test_score"].std()

    print(f"Train Accuracy:      {train_mean:.5f}")
    print(f"Validation Accuracy: {val_mean:.5f}")
    print(f"Validation Std:      {val_std:.5f}")

    return val_mean

In [13]:
evaluate_features(X,y,stk)

Train Accuracy:      0.98156
Validation Accuracy: 0.88144
Validation Std:      0.00410


np.float64(0.8814375)

In [14]:
evaluate_features(X_temp,y,stk)

Train Accuracy:      0.98475
Validation Accuracy: 0.87975
Validation Std:      0.00303


np.float64(0.8797499999999999)

In [15]:
X_wear = X.copy()

X_wear["wear_load"] = (
    X_wear["component_wear"] *
    X_wear["mechanical_load"]
)

wear_score = evaluate_features(
    X_wear,
    y,
    stk
)

Train Accuracy:      0.98294
Validation Accuracy: 0.88200
Validation Std:      0.00219


In [16]:
X_stress = X.copy()

X_stress["stress_ratio"] = (
    X_stress["thermal_stress"] /
    X_stress["mechanical_load"]
)

stress_score = evaluate_features(
    X_stress,
    y,
    stk
)

Train Accuracy:      0.98289
Validation Accuracy: 0.88138
Validation Std:      0.00378


In [17]:
X_try = X.copy()

X_try["stress_ratio"] =(
    X_try["thermal_stress"] /
    X_try["mechanical_load"]
)

X_try["wear_load"] = (
    X_try["component_wear"] *
    X_try["mechanical_load"]
)

combo_score = evaluate_features(X_try,y,stk)



Train Accuracy:      0.98455
Validation Accuracy: 0.88181
Validation Std:      0.00194


In [22]:
X_thermal = X.copy()

X_thermal["thermal_severity"] = (
    X_thermal["process_temp"]
    * X_thermal["thermal_stress"]
    / (X_thermal["coolant_flow"] + 1e-6)
)

thermal_score = evaluate_features(X_thermal, y, stk)

Train Accuracy:      0.98283
Validation Accuracy: 0.88094
Validation Std:      0.00190


In [24]:
X_mechanical = X.copy()

X_mechanical["mechanical_severity"] = (
    X_mechanical["component_wear"]
    * X_mechanical["mechanical_load"]
    * X_mechanical["vibration_index"]
)

mechanical_score = evaluate_features(
    X_mechanical, y, stk
)

Train Accuracy:      0.98277
Validation Accuracy: 0.88081
Validation Std:      0.00243


In [25]:
X_both = X.copy()

X_both["thermal_severity"] = (
    X_both["process_temp"]
    * X_both["thermal_stress"]
    / (X_both["coolant_flow"] + 1e-6)
)

X_both["mechanical_severity"] = (
    X_both["component_wear"]
    * X_both["mechanical_load"]
    * X_both["vibration_index"]
)

both_score = evaluate_features(X_both, y, stk)

Train Accuracy:      0.98397
Validation Accuracy: 0.88100
Validation Std:      0.00238


In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.neighbors import KNeighborsClassifier


def evaluate_knn_features(X_data, y, skf):

    cat_cols = X_data.select_dtypes(
        exclude="number"
    ).columns.to_list()

    num_cols = X_data.select_dtypes(
        include="number"
    ).columns.to_list()

    # KNN için scaling önemli
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ])

    model = KNeighborsClassifier()

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    results = cross_validate(
        pipeline,
        X_data,
        y,
        cv=skf,
        scoring="accuracy",
        return_train_score=True
    )

    train_mean = results["train_score"].mean()
    val_mean = results["test_score"].mean()
    val_std = results["test_score"].std()

    print(f"Train Accuracy:      {train_mean:.5f}")
    print(f"Validation Accuracy: {val_mean:.5f}")
    print(f"Validation Std:      {val_std:.5f}")

    return val_mean

In [27]:
knn_baseline = evaluate_knn_features(
    X, y, stk
)

Train Accuracy:      0.89169
Validation Accuracy: 0.84869
Validation Std:      0.00336


In [28]:
X_mechanical = X.copy()

X_mechanical["mechanical_severity"] = (
    X_mechanical["component_wear"]
    * X_mechanical["mechanical_load"]
    * X_mechanical["vibration_index"]
)

knn_mechanical = evaluate_knn_features(
    X_mechanical, y, stk
)

Train Accuracy:      0.89202
Validation Accuracy: 0.84950
Validation Std:      0.00273


In [29]:
X_thermal = X.copy()

X_thermal["thermal_severity"] = (
    X_thermal["process_temp"]
    * X_thermal["thermal_stress"]
    / (X_thermal["coolant_flow"] + 1e-6)
)

knn_thermal = evaluate_knn_features(
    X_thermal, y, stk
)

Train Accuracy:      0.89391
Validation Accuracy: 0.85244
Validation Std:      0.00340


In [30]:
X_try = X.copy()

X_try["temp_per_cooling"] = (
    (X_try["process_temp"] - X_try["ambient_temp"])
    / (X_try["coolant_flow"] + 1e-6)
)

evaluate_knn_features(X_try, y, stk)

Train Accuracy:      0.89475
Validation Accuracy: 0.85425
Validation Std:      0.00310


np.float64(0.8542500000000001)

In [31]:
X_try = X.copy()

X_try["torque_per_speed"] = (
    X_try["drive_torque"]
    / X_try["shaft_speed"]
)

evaluate_knn_features(X_try, y, stk)

Train Accuracy:      0.88703
Validation Accuracy: 0.84594
Validation Std:      0.00414


np.float64(0.8459375)